# 1. 불러오기

In [2]:
from pathlib import Path
import sys

# 현재 위치에서 프로젝트 최상위 폴더 탐색
ROOT = Path.cwd().resolve()

while ROOT.name != "ah07-cancer-subtype":
    if ROOT == ROOT.parent:
        raise FileNotFoundError("프로젝트 최상위 폴더를 찾을 수 없습니다.")
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("프로젝트 경로:", ROOT)

required_files = [
    "03_hotspot_bins.ipynb",
    "04_xgb_feature_E.ipynb",
]

for filename in required_files:
    matches = list((ROOT / "notebooks").rglob(filename))

    if matches:
        print(f"{filename}: 확인 완료")
        print(f"  위치: {matches[0].relative_to(ROOT)}")
    else:
        print(f"{filename}: 파일 없음")

프로젝트 경로: /Users/admin/Documents/ah07-cancer-subtype
03_hotspot_bins.ipynb: 확인 완료
  위치: notebooks/H_data/03_hotspot_bins.ipynb
04_xgb_feature_E.ipynb: 확인 완료
  위치: notebooks/A_model/04_xgb_feature_E.ipynb


In [ ]:
# 기존 노트북의 함수 목록 확인

import json
import ast

notebook_paths = {
    "Hotspot + Position-bin": ROOT / "notebooks/H_data/03_hotspot_bins.ipynb",
    "XGBoost Feature E": ROOT / "notebooks/A_model/04_xgb_feature_E.ipynb",
}

for name, path in notebook_paths.items():
    print(f"\n{'=' * 60}")
    print(name)
    print(f"{'=' * 60}")

    with open(path, encoding="utf-8") as f:
        nb = json.load(f)

    for i, cell in enumerate(nb["cells"]):
        if cell["cell_type"] != "code":
            continue

        source = "".join(cell["source"])

        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue

        definitions = [
            node.name
            for node in tree.body
            if isinstance(node, (ast.FunctionDef, ast.ClassDef))
        ]

        if definitions:
            print(f"셀 {i}: {', '.join(definitions)}")


Hotspot + Position-bin
셀 5: make_features_E, make_features_E_sparse
셀 9: make_hotspot_features
셀 12: make_features_E_hotspot, make_features_E_hotspot_sparse
셀 15: extract_position
셀 16: extract_position_bins
셀 18: PositionBinTransformer
셀 20: FeaturesEWithPositionBins
셀 22: FeaturesEWithHotspotAndPositionBins
셀 24: make_features_E_hotspot3_sparse
셀 26: FeaturesEWithHotspot3AndPositionBins
셀 32: predict_test_5fold

XGBoost Feature E
셀 1: select_xgb_device
셀 4: feature_frame, make_E


In [4]:
# STEP 3. 기존 피처 생성기 구현 확인

import json

path = ROOT / "notebooks/H_data/03_hotspot_bins.ipynb"

with open(path, encoding="utf-8") as f:
    nb = json.load(f)

# 필요한 셀의 코드 출력
for cell_idx in [18, 20, 22, 24, 26]:
    cell = nb["cells"][cell_idx]
    source = "".join(cell["source"])

    print(f"\n{'=' * 70}")
    print(f"셀 {cell_idx}")
    print(f"{'=' * 70}")
    print(source)


셀 18
# Fold별 Position-bin 변환기 구현

from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import csr_matrix
from collections import Counter

class PositionBinTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, bin_size=50, min_patients=5):
        self.bin_size = bin_size
        self.min_patients = min_patients

    def fit(self, X, y=None):
        _, cols, _ = extract_position_bins(
            X, bin_size=self.bin_size
        )

        counts = Counter(cols)

        self.selected_bins_ = sorted(
            feature
            for feature, count in counts.items()
            if count >= self.min_patients
        )

        self.bin_to_idx_ = {
            feature: idx
            for idx, feature in enumerate(self.selected_bins_)
        }

        return self

    def transform(self, X):
        rows, cols, _ = extract_position_bins(
            X, bin_size=self.bin_size
        )

        selected_rows = []
        selected_cols = []

     

In [5]:
# Position-bin 빈도 계산 검증

# STEP 4. Position-bin의 환자별 중복 집계 확인
import json
import ast

path = ROOT / "notebooks/H_data/03_hotspot_bins.ipynb"

with open(path, encoding="utf-8") as f:
    nb = json.load(f)

# extract_position_bins 함수 구현 확인
for cell_idx in [15, 16]:
    source = "".join(nb["cells"][cell_idx]["source"])

    print(f"\n{'=' * 65}")
    print(f"셀 {cell_idx}")
    print(f"{'=' * 65}")
    print(source)


셀 15
#  변이 위치 추출 방식 확인

import re

def extract_position(variant):
    match = re.search(r"\d+", variant)
    return int(match.group()) if match else None

examples = [
    "V600E",
    "R132H",
    "WQ288fs",
    "-287fs",
    "R213*",
    "WT",
    "V600E V600E",
]

for variant in examples:
    tokens = variant.split()
    positions = [extract_position(token) for token in tokens]
    print(f"{variant:15} → {positions}")

셀 16
# Position-bin 후보 생성 함수

from collections import defaultdict

def extract_position_bins(X, bin_size=50):
    rows = []
    cols = []
    gene_bins = set()

    for row_idx, (_, row) in enumerate(X.iterrows()):
        patient_bins = set()

        for gene, value in row.items():
            if gene == "ID":
                continue

            if pd.isna(value) or str(value) == "WT":
                continue

            for variant in str(value).split():
                pos = extract_position(variant)

                if pos is not None:
                    fe

In [6]:
# STEP 5. Feature E 및 Hotspot 생성 코드 확인

import json

notebooks_to_check = {
    "Feature E": (
        ROOT / "notebooks/H_data/03_hotspot_bins.ipynb",
        [5]
    ),
    "Hotspot": (
        ROOT / "notebooks/H_data/03_hotspot_bins.ipynb",
        [8, 9]
    ),
}

for name, (path, cell_indices) in notebooks_to_check.items():
    with open(path, encoding="utf-8") as f:
        nb = json.load(f)

    for cell_idx in cell_indices:
        source = "".join(nb["cells"][cell_idx]["source"])

        print(f"\n{'=' * 70}")
        print(f"{name} | 셀 {cell_idx}")
        print(f"{'=' * 70}")
        print(source)


Feature E | 셀 5
# 기준 모델 실행

# 피처 구성 E — 아미노산 치환 쌍 + 유전자별 끊김형 피처

def make_features_E(X):
    X_binary = binarize_mutations(X)

    X_count = np.log1p(mutation_count(X))
    X_types = np.log1p(mutation_type_counts(X))
    X_pairs = np.log1p(amino_acid_pair_counts(X))
    X_trunc = gene_truncation_flags(X)

    return pd.concat(
        [X_binary, X_count, X_types, X_pairs, X_trunc],
        axis=1
    )


def make_features_E_sparse(X):
    return csr_matrix(
        make_features_E(X).to_numpy(dtype=np.float32)
    )


pipeline_E_LR = Pipeline([
    ("features", FunctionTransformer(make_features_E_sparse)),
    ("model", LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=3000,
        solver="lbfgs"
    ))
])

cv_E_LR = run_cv(
    pipeline_E_LR,
    X,
    y,
    folds=folds
)

Hotspot | 셀 8
genes = ["IDH1", "BRAF", "TP53", "NPM1"]

for gene in genes:
    print(f"\n[{gene}]")
    if gene in X.columns:
        print(X[gene].dropna().value_counts().head

In [7]:
# Hotspot 정의 및 Feature E 함수 위치 확인

# STEP 6. HOTSPOTS 정의와 Feature E 의존 함수 위치 확인

import json
import ast

path = ROOT / "notebooks/H_data/03_hotspot_bins.ipynb"

with open(path, encoding="utf-8") as f:
    nb = json.load(f)

targets = {
    "HOTSPOTS",
    "binarize_mutations",
    "mutation_count",
    "mutation_type_counts",
    "amino_acid_pair_counts",
    "gene_truncation_flags",
}

for i, cell in enumerate(nb["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    found = []

    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.ClassDef)):
            if node.name in targets:
                found.append(node.name)

        elif isinstance(node, (ast.Assign, ast.AnnAssign)):
            names = (
                [t.id for t in node.targets if isinstance(t, ast.Name)]
                if isinstance(node, ast.Assign)
                else [node.target.id] if isinstance(node.target, ast.Name) else []
            )
            found.extend(name for name in names if name in targets)

    if found:
        print(f"\n{'=' * 65}")
        print(f"셀 {i} | 발견: {', '.join(found)}")
        print(f"{'=' * 65}")
        print(source)


셀 7 | 발견: HOTSPOTS
# 유전자별 Hotspot 아미노산 위치
HOTSPOTS = {
    "IDH1": ["R132"],
    "IDH2": ["R140", "R172"],
    "BRAF": ["V600"],
    "HRAS": ["Q61", "G13", "G12"],
    "KIT": ["D816", "N822", "K642"],
    "CTNNB1": ["S33", "S37", "S45", "T41", "G34", "D32"],
    "U2AF1": ["S34"],
    "PIK3CA": ["E545", "H1047", "E542"],
    "EGFR": ["L858", "G598", "A289"],
    "RET": ["M918"],
    "RAC1": ["P29"],
    "SPOP": ["F133", "W131", "F102", "Y87"],
    "EZH2": ["Y641", "Y646"],
    "TP53": ["R175", "R248", "R273"],
    "PTEN": ["R130"],
}

# 일반 Hotspot 35개 + NPM1 frameshift 1개
assert sum(len(v) for v in HOTSPOTS.values()) == 35

print("HOTSPOTS 정의 완료")


n_hotspots = sum(len(v) for v in HOTSPOTS.values()) + 1

print("일반 Hotspot:", sum(len(v) for v in HOTSPOTS.values()))
print("NPM1 frameshift:", 1)
print("전체 Hotspot 피처 수:", n_hotspots)

assert n_hotspots == 36


In [8]:
# STEP 7. 기존 노트북의 import 및 데이터 로딩 코드 확인

import json

notebook_paths = {
    "Hotspot + Position-bin": ROOT / "notebooks/H_data/03_hotspot_bins.ipynb",
    "XGBoost Feature E": ROOT / "notebooks/A_model/04_xgb_feature_E.ipynb",
}

for name, path in notebook_paths.items():
    with open(path, encoding="utf-8") as f:
        nb = json.load(f)

    print(f"\n{'=' * 70}")
    print(name)
    print(f"{'=' * 70}")

    for cell_idx in [0, 1, 2, 3]:
        if cell_idx >= len(nb["cells"]):
            continue

        cell = nb["cells"][cell_idx]

        if cell["cell_type"] != "code":
            continue

        print(f"\n--- 셀 {cell_idx} ---")
        print("".join(cell["source"]))


Hotspot + Position-bin

--- 셀 2 ---
import sys
from pathlib import Path

# 프로젝트 루트 경로
PROJECT_ROOT = Path.cwd().resolve().parents[1]

# Python 모듈 검색 경로에 추가
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("프로젝트 루트:", PROJECT_ROOT)
print("src 폴더 존재:", (PROJECT_ROOT / "src").is_dir())

--- 셀 3 ---
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression

from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import binarize_mutations
from src.features import (
    mutation_count,
    mutation_type_counts,
    amino_acid_pair_counts,
    gene_truncation_flags,
)
from src.validation import get_folds, run_cv

print("Imports completed")

XGBoost Feature E

--- 셀 1 ---
from pathlib import Path
import sys, inspect, json, warnings
import numpy as np
import pandas a

In [9]:
# STEP 8. 데이터 로딩 및 고정 Holdout 확인

import numpy as np
import pandas as pd

from src.config import DATA_DIR, SEED, set_seed
from src.data_loader import load_data, split_xy, encode_target
from src.validation import get_folds

set_seed()

# 원본 학습 데이터 로딩
train, test = load_data(DATA_DIR)

X_raw, y_raw = split_xy(train)
y, le = encode_target(y_raw)

print("전체 학습 데이터:", X_raw.shape)
print("클래스 수:", len(le.classes_))

# 고정 Holdout 관리 파일 존재 여부 확인
manifest_path = ROOT / "scripts" / "fixed_holdout_manifest.py"

print("\n고정 Holdout 관리 파일:")
print("경로:", manifest_path)
print("존재 여부:", manifest_path.is_file())

# 피처 생성 함수 확인
from src.preprocessing import binarize_mutations
from src.features import (
    mutation_count,
    mutation_type_counts,
    amino_acid_pair_counts,
    gene_truncation_flags,
)

print("\nFeature E 함수 import 완료")

전체 학습 데이터: (6201, 4384)
클래스 수: 26

고정 Holdout 관리 파일:
경로: /Users/admin/Documents/ah07-cancer-subtype/scripts/fixed_holdout_manifest.py
존재 여부: True

Feature E 함수 import 완료


In [10]:
# STEP 9. 고정 Holdout 분리 및 개발 데이터 CV 구성

from scripts import fixed_holdout_manifest as holdout_module

# 팀에서 고정한 인덱스 불러오기
dev_idx, hold_idx = holdout_module.verify()

# 개발 데이터만 추출
X_dev_raw = X_raw.iloc[dev_idx].reset_index(drop=True)
y_dev = np.asarray(y)[dev_idx]

# Holdout은 크기만 확인하고 실험에 사용하지 않음
print("전체 데이터:", len(X_raw))
print("개발 데이터:", len(X_dev_raw))
print("고정 Holdout:", len(hold_idx))

# 개발 데이터에 대해서만 StratifiedGroupKFold 구성
folds_dev = get_folds(
    y_dev,
    X=X_dev_raw,
    classes=le.classes_
)

print("\nCV Fold 수:", len(folds_dev))

for i, (train_idx, val_idx) in enumerate(folds_dev, start=1):
    print(
        f"Fold {i}: "
        f"train={len(train_idx)}, "
        f"validation={len(val_idx)}"
    )

# 데이터 분리 검증
assert len(set(dev_idx) & set(hold_idx)) == 0
assert len(dev_idx) + len(hold_idx) == len(X_raw)
assert len(folds_dev) == 5

print("\n개발 데이터 / Holdout 분리 완료")

VERIFIED: dev=4960, holdout=1241; same IDs and row signatures
전체 데이터: 6201
개발 데이터: 4960
고정 Holdout: 1241

CV Fold 수: 5
Fold 1: train=3968, validation=992
Fold 2: train=3968, validation=992
Fold 3: train=3968, validation=992
Fold 4: train=3968, validation=992
Fold 5: train=3968, validation=992

개발 데이터 / Holdout 분리 완료


In [11]:
# STEP 10. 기존 Feature E + Hotspot ×3 + Position-bin 코드 연결

import ast
import json
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix, hstack
from sklearn.base import BaseEstimator, TransformerMixin

source_path = ROOT / "notebooks/H_data/03_hotspot_bins.ipynb"

with open(source_path, encoding="utf-8") as f:
    source_nb = json.load(f)

# 가져올 함수와 클래스만 지정
required_names = [
    "HOTSPOTS",
    "make_features_E",
    "make_features_E_sparse",
    "make_hotspot_features",
    "extract_position",
    "extract_position_bins",
    "PositionBinTransformer",
    "FeaturesEWithHotspotAndPositionBins",
    "FeaturesEWithHotspot3AndPositionBins",
]

loaded_names = []

for name in required_names:
    found = False

    for cell in source_nb["cells"]:
        if cell["cell_type"] != "code":
            continue

        source = "".join(cell["source"])

        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue

        for node in tree.body:
            is_definition = (
                isinstance(node, (ast.FunctionDef, ast.ClassDef))
                and node.name == name
            )

            is_assignment = (
                isinstance(node, ast.Assign)
                and any(
                    isinstance(target, ast.Name) and target.id == name
                    for target in node.targets
                )
            )

            if is_definition or is_assignment:
                module = ast.Module(body=[node], type_ignores=[])
                ast.fix_missing_locations(module)

                exec(
                    compile(module, str(source_path), "exec"),
                    globals()
                )

                loaded_names.append(name)
                found = True
                break

        if found:
            break

    if not found:
        raise RuntimeError(f"정의를 찾을 수 없습니다: {name}")

print("불러온 정의:")
for name in loaded_names:
    print(" -", name)

print("\nHotspot 개수:", sum(map(len, HOTSPOTS.values())) + 1)

# 개발 데이터 일부로 피처 생성 확인
X_sample = X_dev_raw.head(3)

sample_E = make_features_E_sparse(X_sample)
sample_hotspot = make_hotspot_features(X_sample)

print("\nFeature E 샘플 shape:", sample_E.shape)
print("Hotspot 샘플 shape:", sample_hotspot.shape)

assert sample_E.shape[0] == 3
assert sample_hotspot.shape == (3, 36)

print("\n기존 피처 생성 함수 연결 완료")

불러온 정의:
 - HOTSPOTS
 - make_features_E
 - make_features_E_sparse
 - make_hotspot_features
 - extract_position
 - extract_position_bins
 - PositionBinTransformer
 - FeaturesEWithHotspotAndPositionBins
 - FeaturesEWithHotspot3AndPositionBins

Hotspot 개수: 36

Feature E 샘플 shape: (3, 9174)
Hotspot 샘플 shape: (3, 36)

기존 피처 생성 함수 연결 완료


# 2. 신규 피처 5개 추가 변환기 구현

In [12]:
# STEP 11. 신장암 / 뇌종양 신규 피처 추가

def make_subtype_features(X):
    """원본 유전자 변이 정보로부터 신규 이진 피처 5개 생성"""

    genes = ["VHL", "TP53", "MET", "IDH1", "ATRX", "PTEN", "EGFR"]

    mut = {
        g: (X[g].notna() & X[g].ne("WT")).astype("int8")
        for g in genes
    }

    features = pd.DataFrame(index=X.index)

    # 신장암 관련 피처 2개
    features["kidney_vhl0_tp53_1"] = (
        (1 - mut["VHL"]) * mut["TP53"]
    )
    features["kidney_vhl0_met_1"] = (
        (1 - mut["VHL"]) * mut["MET"]
    )

    # 뇌종양 관련 피처 3개
    features["brain_idh1_0_atrx_1"] = (
        (1 - mut["IDH1"]) * mut["ATRX"]
    )
    features["brain_idh1_0_pten_1"] = (
        (1 - mut["IDH1"]) * mut["PTEN"]
    )
    features["brain_egfr_0_pten_1"] = (
        (1 - mut["EGFR"]) * mut["PTEN"]
    )

    return features


class FeaturesEHotspot3PositionSubtype(
    FeaturesEWithHotspot3AndPositionBins
):
    def __init__(
        self,
        bin_size=50,
        min_patients=5,
        subtype_group="all"
    ):
        super().__init__(
            bin_size=bin_size,
            min_patients=min_patients
        )
        self.subtype_group = subtype_group

    def transform(self, X):
        # 기존 E + Hotspot ×3 + Position-bin
        X_base = super().transform(X)

        # 신규 피처 생성
        subtype = make_subtype_features(X)

        if self.subtype_group == "kidney":
            selected = subtype.columns[:2]

        elif self.subtype_group == "brain":
            selected = subtype.columns[2:]

        elif self.subtype_group == "all":
            selected = subtype.columns

        else:
            raise ValueError(
                f"지원하지 않는 subtype_group: {self.subtype_group}"
            )

        X_subtype = csr_matrix(
            subtype[selected].to_numpy(dtype=np.float32)
        )

        return hstack(
            [X_base, X_subtype],
            format="csr",
            dtype=np.float32
        )


# 신규 피처 생성 확인
sample_subtype = make_subtype_features(X_dev_raw.head(3))

print("신규 피처 shape:", sample_subtype.shape)
print("신규 피처 목록:")
print(sample_subtype.columns.tolist())

assert sample_subtype.shape == (3, 5)

print("\n신규 피처 변환기 정의 완료")

신규 피처 shape: (3, 5)
신규 피처 목록:
['kidney_vhl0_tp53_1', 'kidney_vhl0_met_1', 'brain_idh1_0_atrx_1', 'brain_idh1_0_pten_1', 'brain_egfr_0_pten_1']

신규 피처 변환기 정의 완료


In [13]:
# STEP 12. 네 가지 피처 구성 검증

import numpy as np

# 첫 번째 Fold만 사용
train_idx, val_idx = folds_dev[0]

X_fold_train = X_dev_raw.iloc[train_idx]
X_fold_val = X_dev_raw.iloc[val_idx]

# Baseline 변환기
baseline_transformer = FeaturesEWithHotspot3AndPositionBins(
    bin_size=50,
    min_patients=5
)

baseline_transformer.fit(X_fold_train)

X_base_train = baseline_transformer.transform(X_fold_train)
X_base_val = baseline_transformer.transform(X_fold_val)

print("=== Baseline ===")
print("Train:", X_base_train.shape)
print("Validation:", X_base_val.shape)
print(
    "선택된 Position-bin:",
    len(baseline_transformer.position_transformer_.selected_bins_)
)

# 신규 피처 추가 실험
experiments = {
    "D_kidney": ("kidney", 2),
    "E_brain": ("brain", 3),
    "F_all": ("all", 5),
}

for name, (group, n_features) in experiments.items():

    transformer = FeaturesEHotspot3PositionSubtype(
        bin_size=50,
        min_patients=5,
        subtype_group=group
    )

    transformer.fit(X_fold_train)

    X_train_new = transformer.transform(X_fold_train)
    X_val_new = transformer.transform(X_fold_val)

    # 피처 개수 검증
    assert X_train_new.shape[1] == X_base_train.shape[1] + n_features
    assert X_val_new.shape[1] == X_base_val.shape[1] + n_features

    # 기존 피처가 변경되지 않았는지 확인
    assert (X_train_new[:, :X_base_train.shape[1]] != X_base_train).nnz == 0
    assert (X_val_new[:, :X_base_val.shape[1]] != X_base_val).nnz == 0

    print(f"\n=== {name} ===")
    print("Train:", X_train_new.shape)
    print("Validation:", X_val_new.shape)
    print("추가 피처:", n_features)

print("\n모든 피처 구성 검증 완료")

=== Baseline ===
Train: (3968, 20686)
Validation: (992, 20686)
선택된 Position-bin: 11476

=== D_kidney ===
Train: (3968, 20688)
Validation: (992, 20688)
추가 피처: 2

=== E_brain ===
Train: (3968, 20689)
Validation: (992, 20689)
추가 피처: 3

=== F_all ===
Train: (3968, 20691)
Validation: (992, 20691)
추가 피처: 5

모든 피처 구성 검증 완료


In [ ]:
# STEP 13. XGBoost 모델 및 Pipeline 설정

from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

# 기존 Feature E 실험과 동일한 모델 설정
xgb_params = dict(
    n_estimators=250,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=len(le.classes_),
    eval_metric="mlogloss",
    tree_method="hist",
    device="cpu",
    random_state=SEED,
    n_jobs=-1,
)

def make_experiment_pipeline(group=None):

    if group is None:
        transformer = FeaturesEWithHotspot3AndPositionBins(
            bin_size=50,
            min_patients=5
        )
    else:
        transformer = FeaturesEHotspot3PositionSubtype(
            bin_size=50,
            min_patients=5,
            subtype_group=group
        )

    return Pipeline([
        ("features", transformer),
        ("model", XGBClassifier(**xgb_params))
    ])


# 네 가지 실험 정의
pipelines = {
    "Baseline": make_experiment_pipeline(),
    "D_kidney": make_experiment_pipeline("kidney"),
    "E_brain": make_experiment_pipeline("brain"),
    "F_all": make_experiment_pipeline("all"),
}

for name, pipeline in pipelines.items():
    print(
        f"{name}: "
        f"{pipeline.named_steps['features'].__class__.__name__}"
        f" + {pipeline.named_steps['model'].__class__.__name__}"
    )

print("\nXGBoost 설정 완료")

Baseline: FeaturesEWithHotspot3AndPositionBins + XGBClassifier
D_kidney: FeaturesEHotspot3PositionSubtype + XGBClassifier
E_brain: FeaturesEHotspot3PositionSubtype + XGBClassifier
F_all: FeaturesEHotspot3PositionSubtype + XGBClassifier

XGBoost 설정 완료


# 3. E + Hotspot ×3 + Position-bin을 기준으로 신규 파생 피처 5개 추가

In [15]:
# STEP 14. 5-Fold CV 성능 비교

import time
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import f1_score

cv_results = {}
oof_predictions = {}

for exp_name, pipeline in pipelines.items():

    print(f"\n{'=' * 60}", flush=True)
    print(f"실험: {exp_name}", flush=True)
    print(f"{'=' * 60}", flush=True)

    fold_scores = []
    oof_pred = np.full(len(y_dev), -1, dtype=int)

    start_time = time.time()

    for fold_num, (train_idx, val_idx) in enumerate(folds_dev, start=1):

        X_train = X_dev_raw.iloc[train_idx]
        X_val = X_dev_raw.iloc[val_idx]

        y_train = y_dev[train_idx]
        y_val = y_dev[val_idx]

        # Fold마다 독립적인 Pipeline 생성
        model = clone(pipeline)

        # Position-bin fit은 학습 Fold에서만 수행
        model.fit(X_train, y_train)

        pred = model.predict(X_val).astype(int)

        score = f1_score(
            y_val,
            pred,
            average="macro",
            labels=np.arange(len(le.classes_)),
            zero_division=0
        )

        fold_scores.append(score)
        oof_pred[val_idx] = pred

        print(
            f"Fold {fold_num}/5 | Macro F1: {score:.4f}",
            flush=True
        )

    assert np.all(oof_pred >= 0), "OOF 예측 누락 발생"

    oof_score = f1_score(
        y_dev,
        oof_pred,
        average="macro",
        labels=np.arange(len(le.classes_)),
        zero_division=0
    )

    elapsed = time.time() - start_time

    cv_results[exp_name] = {
        "cv_mean": np.mean(fold_scores),
        "cv_std": np.std(fold_scores),
        "oof_macro_f1": oof_score,
        "elapsed_sec": elapsed,
    }

    oof_predictions[exp_name] = oof_pred

    print(
        f"\n{exp_name} 완료 | "
        f"CV Mean: {np.mean(fold_scores):.4f} | "
        f"OOF: {oof_score:.4f} | "
        f"소요 시간: {elapsed:.1f}초",
        flush=True
    )


# 결과 비교
results_df = pd.DataFrame(cv_results).T

baseline_score = results_df.loc["Baseline", "cv_mean"]

results_df["delta_vs_baseline"] = (
    results_df["cv_mean"] - baseline_score
)

print("\n========== 최종 CV 비교 ==========")

display(
    results_df[
        ["cv_mean", "cv_std", "oof_macro_f1", "delta_vs_baseline", "elapsed_sec"]
    ].round(4)
)


실험: Baseline
Fold 1/5 | Macro F1: 0.4758
Fold 2/5 | Macro F1: 0.4422
Fold 3/5 | Macro F1: 0.4732
Fold 4/5 | Macro F1: 0.5017
Fold 5/5 | Macro F1: 0.4689

Baseline 완료 | CV Mean: 0.4724 | OOF: 0.4774 | 소요 시간: 351.8초

실험: D_kidney
Fold 1/5 | Macro F1: 0.4665
Fold 2/5 | Macro F1: 0.4536
Fold 3/5 | Macro F1: 0.4628
Fold 4/5 | Macro F1: 0.4793
Fold 5/5 | Macro F1: 0.4625

D_kidney 완료 | CV Mean: 0.4649 | OOF: 0.4692 | 소요 시간: 350.1초

실험: E_brain
Fold 1/5 | Macro F1: 0.4767
Fold 2/5 | Macro F1: 0.4427
Fold 3/5 | Macro F1: 0.4644
Fold 4/5 | Macro F1: 0.4842
Fold 5/5 | Macro F1: 0.4660

E_brain 완료 | CV Mean: 0.4668 | OOF: 0.4709 | 소요 시간: 348.0초

실험: F_all
Fold 1/5 | Macro F1: 0.4798
Fold 2/5 | Macro F1: 0.4507
Fold 3/5 | Macro F1: 0.4641
Fold 4/5 | Macro F1: 0.4831
Fold 5/5 | Macro F1: 0.4684

F_all 완료 | CV Mean: 0.4692 | OOF: 0.4734 | 소요 시간: 349.2초

========== 최종 CV 비교 ==========


,cv_mean,cv_std,oof_macro_f1,delta_vs_baseline,elapsed_sec
Baseline,0.4724,0.0189,0.4774,0.0000,351.8080
D_kidney,0.4649,0.0083,0.4692,-0.0074,350.0904
E_brain,0.4668,0.0141,0.4709,-0.0056,348.0314
F_all,0.4692,0.0116,0.4734,-0.0031,349.1636


In [16]:
# STEP 15. 주요 4개 암종 F1 / Recall 비교

from sklearn.metrics import f1_score, recall_score
import numpy as np
import pandas as pd

target_classes = ["KIPAN", "KIRC", "GBMLGG", "LGG"]

class_names = le.classes_

rows = []

for exp_name, pred in oof_predictions.items():

    for class_name in target_classes:

        class_idx = np.where(class_names == class_name)[0][0]

        f1 = f1_score(
            y_dev == class_idx,
            pred == class_idx,
            zero_division=0
        )

        recall = recall_score(
            y_dev == class_idx,
            pred == class_idx,
            zero_division=0
        )

        rows.append({
            "experiment": exp_name,
            "class": class_name,
            "f1": f1,
            "recall": recall
        })

class_results = pd.DataFrame(rows)

baseline_f1 = (
    class_results[class_results["experiment"] == "Baseline"]
    .set_index("class")["f1"]
)

class_results["delta_f1"] = class_results.apply(
    lambda row: row["f1"] - baseline_f1[row["class"]],
    axis=1
)

print("=== 주요 4개 암종 성능 비교 ===")

display(
    class_results.round(4)
)

=== 주요 4개 암종 성능 비교 ===


,experiment,class,f1,recall,delta_f1
0,Baseline,KIPAN,0.4004,0.4927,0.0000
1,Baseline,KIRC,0.2897,0.2322,0.0000
2,Baseline,GBMLGG,0.5006,0.5285,0.0000
3,Baseline,LGG,0.4740,0.4481,0.0000
4,D_kidney,KIPAN,0.4125,0.5121,0.0121
5,D_kidney,KIRC,0.2567,0.1985,-0.0331
6,D_kidney,GBMLGG,0.4974,0.5203,-0.0032
7,D_kidney,LGG,0.4491,0.4098,-0.0249
8,E_brain,KIPAN,0.3984,0.4854,-0.0020
9,E_brain,KIRC,0.2891,0.2285,-0.0006


- GBMLGG의 개선이 LGG의 성능 하락을 동반하는 양상임.

In [17]:
# STEP 16. 주요 혼동 쌍의 오분류 변화

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

class_names = list(le.classes_)

pairs = [
    ("KIPAN", "KIRC"),
    ("KIRC", "KIPAN"),
    ("GBMLGG", "LGG"),
    ("LGG", "GBMLGG"),
]

rows = []

for exp_name, pred in oof_predictions.items():

    cm = confusion_matrix(
        y_dev,
        pred,
        labels=np.arange(len(class_names))
    )

    for true_class, pred_class in pairs:
        true_idx = class_names.index(true_class)
        pred_idx = class_names.index(pred_class)

        rows.append({
            "experiment": exp_name,
            "true_class": true_class,
            "predicted_class": pred_class,
            "misclassified_count": int(cm[true_idx, pred_idx])
        })

confusion_results = pd.DataFrame(rows)

baseline_counts = (
    confusion_results[
        confusion_results["experiment"] == "Baseline"
    ]
    .set_index(["true_class", "predicted_class"])["misclassified_count"]
)

confusion_results["delta_vs_baseline"] = confusion_results.apply(
    lambda row: (
        row["misclassified_count"]
        - baseline_counts.loc[
            (row["true_class"], row["predicted_class"])
        ]
    ),
    axis=1
)

display(confusion_results)

,experiment,true_class,predicted_class,misclassified_count,delta_vs_baseline
0,Baseline,KIPAN,KIRC,78,0
1,Baseline,KIRC,KIPAN,134,0
2,Baseline,GBMLGG,LGG,78,0
3,Baseline,LGG,GBMLGG,88,0
4,D_kidney,KIPAN,KIRC,71,-7
5,D_kidney,KIRC,KIPAN,142,8
6,D_kidney,GBMLGG,LGG,73,-5
7,D_kidney,LGG,GBMLGG,90,2
8,E_brain,KIPAN,KIRC,75,-3
9,E_brain,KIRC,KIPAN,133,-1


- 신규 피처가 특정 방향의 오분류를 줄이기는 했지만, 반대 방향의 오분류가 늘어나는 문제가 있고 전체 Macro F1도 개선하지 못했음.

# 4. E + Position-bin을 기준으로 신규 파생 피처 5개 추가

## 4-1. 불러오기 및 변환기 생성

In [22]:
# 3차 실험 STEP 1. 기존 변수 및 클래스 확인
import inspect

names = [
    "X_dev_raw",
    "y_dev",
    "folds_dev",
    "pipelines",
    "FeaturesEWithHotspotAndPositionBins",
    "FeaturesEWithHotspot3AndPositionBins",
    "FeaturesEHotspot3PositionSubtype",
    "make_subtype_features",
]

for name in names:
    obj = globals().get(name)

    if obj is None:
        print(f"{name}: 없음")
    elif name == "pipelines":
        print(f"{name}: {list(obj.keys())}")
    elif name == "folds_dev":
        print(f"{name}: {len(obj)} folds")
    elif name in ("X_dev_raw", "y_dev"):
        print(f"{name}: {getattr(obj, 'shape', None)}")
    else:
        print(f"{name}: 확인됨")

print("\n=== 기존 Pipeline 구성 ===")

if "pipelines" in globals():
    for name, pipe in pipelines.items():
        print(f"\n[{name}]")
        print(pipe)

X_dev_raw: (4960, 4384)
y_dev: (4960,)
folds_dev: 5 folds
pipelines: ['Baseline', 'D_kidney', 'E_brain', 'F_all']
FeaturesEWithHotspotAndPositionBins: 확인됨
FeaturesEWithHotspot3AndPositionBins: 확인됨
FeaturesEHotspot3PositionSubtype: 확인됨
make_subtype_features: 확인됨

=== 기존 Pipeline 구성 ===

[Baseline]
Pipeline(steps=[('features', FeaturesEWithHotspot3AndPositionBins()),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=0.8, device='cpu',
                               early_stopping_rounds=None,
                               enable_categorical=True, eval_metric='mlogloss',
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=No

In [24]:
# 3차 실험 STEP 2 수정
# 기존 피처 클래스의 구조 및 메서드 확인

import inspect

classes = [
    FeaturesEWithHotspotAndPositionBins,
    FeaturesEWithHotspot3AndPositionBins,
    FeaturesEHotspot3PositionSubtype
]

for cls in classes:
    print(f"\n{'=' * 60}")
    print(f"클래스: {cls.__name__}")

    print("\n상속 관계:")
    print([c.__name__ for c in cls.__mro__])

    print("\n생성자 파라미터:")
    print(inspect.signature(cls))

    print("\n직접 정의된 메서드:")
    print([
        name for name, value in cls.__dict__.items()
        if callable(value) and not name.startswith("__")
    ])

    print("\n전체 주요 메서드:")
    print([
        name for name in ["fit", "transform", "fit_transform"]
        if hasattr(cls, name)
    ])


클래스: FeaturesEWithHotspotAndPositionBins

상속 관계:
['FeaturesEWithHotspotAndPositionBins', 'BaseEstimator', 'ReprHTMLMixin', '_HTMLDocumentationLinkMixin', '_MetadataRequester', 'TransformerMixin', '_SetOutputMixin', 'object']

생성자 파라미터:
(bin_size=50, min_patients=5)

직접 정의된 메서드:
['fit', 'transform']

전체 주요 메서드:
['fit', 'transform', 'fit_transform']

클래스: FeaturesEWithHotspot3AndPositionBins

상속 관계:
['FeaturesEWithHotspot3AndPositionBins', 'FeaturesEWithHotspotAndPositionBins', 'BaseEstimator', 'ReprHTMLMixin', '_HTMLDocumentationLinkMixin', '_MetadataRequester', 'TransformerMixin', '_SetOutputMixin', 'object']

생성자 파라미터:
(bin_size=50, min_patients=5)

직접 정의된 메서드:
['transform']

전체 주요 메서드:
['fit', 'transform', 'fit_transform']

클래스: FeaturesEHotspot3PositionSubtype

상속 관계:
['FeaturesEHotspot3PositionSubtype', 'FeaturesEWithHotspot3AndPositionBins', 'FeaturesEWithHotspotAndPositionBins', 'BaseEstimator', 'ReprHTMLMixin', '_HTMLDocumentationLinkMixin', '_MetadataRequester', 'TransformerMi

In [25]:
# 3차 실험 STEP 3. 기존 피처 변환기 내부 참조 확인

classes = [
    FeaturesEWithHotspotAndPositionBins,
    FeaturesEWithHotspot3AndPositionBins,
    FeaturesEHotspot3PositionSubtype
]

for cls in classes:
    print(f"\n{'=' * 60}")
    print(f"클래스: {cls.__name__}")

    for method_name in ["fit", "transform"]:
        method = cls.__dict__.get(method_name)

        if method is None:
            print(f"\n{method_name}: 상위 클래스에서 상속")
            continue

        print(f"\n[{method_name}] 참조 이름:")
        print(method.__code__.co_names)

        print(f"\n[{method_name}] 인자:")
        print(method.__code__.co_varnames)


클래스: FeaturesEWithHotspotAndPositionBins

[fit] 참조 이름:
('PositionBinTransformer', 'bin_size', 'min_patients', 'position_transformer_', 'fit')

[fit] 인자:
('self', 'X', 'y')

[transform] 참조 이름:
('isinstance', 'tuple', '_wrap_data_with_container', 'hasattr', 'type', '_make')

[transform] 인자:
('self', 'X', 'args', 'kwargs', 'data_to_wrap', 'return_tuple')

클래스: FeaturesEWithHotspot3AndPositionBins

fit: 상위 클래스에서 상속

[transform] 참조 이름:
('isinstance', 'tuple', '_wrap_data_with_container', 'hasattr', 'type', '_make')

[transform] 인자:
('self', 'X', 'args', 'kwargs', 'data_to_wrap', 'return_tuple')

클래스: FeaturesEHotspot3PositionSubtype

fit: 상위 클래스에서 상속

[transform] 참조 이름:
('isinstance', 'tuple', '_wrap_data_with_container', 'hasattr', 'type', '_make')

[transform] 인자:
('self', 'X', 'args', 'kwargs', 'data_to_wrap', 'return_tuple')


In [26]:
# 3차 실험 STEP 4. scikit-learn 래퍼 내부의 실제 메서드 확인

import dis
import inspect

classes = [
    FeaturesEWithHotspotAndPositionBins,
    FeaturesEWithHotspot3AndPositionBins,
    FeaturesEHotspot3PositionSubtype
]

for cls in classes:
    print(f"\n{'=' * 60}")
    print(f"클래스: {cls.__name__}")

    method = cls.__dict__["transform"]

    # scikit-learn 래퍼를 제거하고 원본 함수 확인
    original = inspect.unwrap(method)

    print("\n실제 transform() 참조 이름:")
    print(original.__code__.co_names)

    print("\n실제 transform() 지역 변수:")
    print(original.__code__.co_varnames)

    print("\n실제 transform() 실행 구조:")
    dis.dis(original)


클래스: FeaturesEWithHotspotAndPositionBins

실제 transform() 참조 이름:
('make_features_E_sparse', 'csr_matrix', 'make_hotspot_features', 'to_numpy', 'np', 'float32', 'position_transformer_', 'transform', 'hstack')

실제 transform() 지역 변수:
('self', 'X', 'X_base', 'X_hotspot', 'X_position')

실제 transform() 실행 구조:
 16           RESUME                   0

 18           LOAD_GLOBAL              1 (make_features_E_sparse + NULL)
              LOAD_FAST_BORROW         1 (X)
              CALL                     1
              STORE_FAST               2 (X_base)

 21           LOAD_GLOBAL              3 (csr_matrix + NULL)

 22           LOAD_GLOBAL              5 (make_hotspot_features + NULL)
              LOAD_FAST_BORROW         1 (X)
              CALL                     1
              LOAD_ATTR                7 (to_numpy + NULL|self)
              LOAD_GLOBAL              8 (np)
              LOAD_ATTR               10 (float32)
              LOAD_CONST               0 (('dtype',))
        

In [27]:
# 3차 실험 STEP 5. E + Position-bin 기반 피처 변환기

import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.base import BaseEstimator, TransformerMixin


class FeaturesEPositionBins(BaseEstimator, TransformerMixin):

    def __init__(self, bin_size=50, min_patients=5):
        self.bin_size = bin_size
        self.min_patients = min_patients

    def fit(self, X, y=None):

        self.position_transformer_ = PositionBinTransformer(
            bin_size=self.bin_size,
            min_patients=self.min_patients
        )

        self.position_transformer_.fit(X, y)

        return self

    def transform(self, X):

        X_base = make_features_E_sparse(X)

        X_position = self.position_transformer_.transform(X)

        return hstack(
            [X_base, X_position],
            format="csr",
            dtype=np.float32
        )


class FeaturesEPositionBinsSubtype(FeaturesEPositionBins):

    def __init__(
        self,
        bin_size=50,
        min_patients=5,
        subtype_group="all"
    ):
        super().__init__(
            bin_size=bin_size,
            min_patients=min_patients
        )
        self.subtype_group = subtype_group

    def transform(self, X):

        X_base = super().transform(X)

        subtype = make_subtype_features(X)

        if self.subtype_group == "kidney":
            selected = subtype.columns[:2]

        elif self.subtype_group == "brain":
            selected = subtype.columns[2:]

        elif self.subtype_group == "all":
            selected = subtype.columns

        else:
            raise ValueError(
                f"지원하지 않는 subtype_group: {self.subtype_group}"
            )

        X_subtype = csr_matrix(
            subtype[selected].to_numpy(dtype=np.float32)
        )

        return hstack(
            [X_base, X_subtype],
            format="csr",
            dtype=np.float32
        )


print("E + Position-bin 변환기 생성 완료")
print("신규 피처 추가 변환기 생성 완료")

E + Position-bin 변환기 생성 완료
신규 피처 추가 변환기 생성 완료


In [28]:
# 3차 실험 STEP 6. Pipeline 구성 및 피처 개수 검증

from sklearn.base import clone
from sklearn.pipeline import Pipeline

# 기존 2차 실험의 XGBoost 설정 재사용
base_model = pipelines["Baseline"].named_steps["model"]

pipelines_pos = {
    "Baseline": Pipeline([
        ("features", FeaturesEPositionBins()),
        ("model", clone(base_model))
    ]),
    "A_kidney": Pipeline([
        ("features", FeaturesEPositionBinsSubtype(
            subtype_group="kidney"
        )),
        ("model", clone(base_model))
    ]),
    "B_brain": Pipeline([
        ("features", FeaturesEPositionBinsSubtype(
            subtype_group="brain"
        )),
        ("model", clone(base_model))
    ]),
    "C_all": Pipeline([
        ("features", FeaturesEPositionBinsSubtype(
            subtype_group="all"
        )),
        ("model", clone(base_model))
    ])
}

# 첫 번째 Fold의 학습 데이터만 사용
train_idx, val_idx = folds_dev[0]

X_train_sample = X_dev_raw.iloc[train_idx]
X_val_sample = X_dev_raw.iloc[val_idx]

print("=== 데이터 크기 ===")
print("Train:", X_train_sample.shape)
print("Validation:", X_val_sample.shape)

print("\n=== 피처 개수 검증 ===")

feature_counts = {}

for name, pipe in pipelines_pos.items():

    transformer = clone(pipe.named_steps["features"])

    # Position-bin은 Train Fold에서만 fit
    transformer.fit(X_train_sample)

    X_train_features = transformer.transform(X_train_sample)
    X_val_features = transformer.transform(X_val_sample)

    feature_counts[name] = X_train_features.shape[1]

    print(
        f"{name:10s} | "
        f"Train: {X_train_features.shape} | "
        f"Val: {X_val_features.shape}"
    )

print("\n=== Baseline 대비 추가 피처 수 ===")

baseline_count = feature_counts["Baseline"]

for name, count in feature_counts.items():
    print(f"{name:10s}: +{count - baseline_count}")

assert feature_counts["A_kidney"] - baseline_count == 2
assert feature_counts["B_brain"] - baseline_count == 3
assert feature_counts["C_all"] - baseline_count == 5

print("\n피처 개수 검증 통과!")

=== 데이터 크기 ===
Train: (3968, 4384)
Validation: (992, 4384)

=== 피처 개수 검증 ===
Baseline   | Train: (3968, 20650) | Val: (992, 20650)
A_kidney   | Train: (3968, 20652) | Val: (992, 20652)
B_brain    | Train: (3968, 20653) | Val: (992, 20653)
C_all      | Train: (3968, 20655) | Val: (992, 20655)

=== Baseline 대비 추가 피처 수 ===
Baseline  : +0
A_kidney  : +2
B_brain   : +3
C_all     : +5

피처 개수 검증 통과!


## 4-2. 3차 실험 5-Fold CV 실행

In [29]:
# 3차 실험 STEP 7. 5-Fold CV 및 OOF 예측

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import f1_score

results_pos = []
oof_preds_pos = {}

n_classes = len(le.classes_)

for name, pipeline in pipelines_pos.items():

    print(f"\n{'=' * 55}")
    print(f"실험: {name}")

    fold_scores = []
    oof_pred = np.full(len(y_dev), -1, dtype=int)

    for fold, (train_idx, val_idx) in enumerate(folds_dev, start=1):

        model = clone(pipeline)

        X_train = X_dev_raw.iloc[train_idx]
        X_val = X_dev_raw.iloc[val_idx]

        y_train = y_dev[train_idx]
        y_val = y_dev[val_idx]

        # Pipeline 내부에서 Position-bin을 Train Fold에만 fit
        model.fit(X_train, y_train)

        pred = model.predict(X_val).astype(int).ravel()

        score = f1_score(
            y_val,
            pred,
            average="macro",
            labels=np.arange(n_classes),
            zero_division=0
        )

        fold_scores.append(score)
        oof_pred[val_idx] = pred

        print(f"Fold {fold}: Macro F1 = {score:.4f}")

    # 모든 Dev 샘플이 정확히 한 번씩 예측됐는지 확인
    assert np.all(oof_pred >= 0), f"{name}: OOF 예측 누락"

    oof_score = f1_score(
        y_dev,
        oof_pred,
        average="macro",
        labels=np.arange(n_classes),
        zero_division=0
    )

    oof_preds_pos[name] = oof_pred

    results_pos.append({
        "experiment": name,
        "cv_mean_macro_f1": np.mean(fold_scores),
        "cv_std": np.std(fold_scores),
        "oof_macro_f1": oof_score
    })

    print(
        f"CV: {np.mean(fold_scores):.4f} "
        f"± {np.std(fold_scores):.4f}"
    )
    print(f"OOF Macro F1: {oof_score:.4f}")


results_pos_df = pd.DataFrame(results_pos)

baseline_score = results_pos_df.loc[
    results_pos_df["experiment"] == "Baseline",
    "cv_mean_macro_f1"
].iloc[0]

results_pos_df["delta_vs_baseline"] = (
    results_pos_df["cv_mean_macro_f1"] - baseline_score
)

print("\n=== 3차 실험 최종 결과 ===")
print(results_pos_df.round(4).to_string(index=False))


실험: Baseline
Fold 1: Macro F1 = 0.4730
Fold 2: Macro F1 = 0.4528
Fold 3: Macro F1 = 0.4674
Fold 4: Macro F1 = 0.4883
Fold 5: Macro F1 = 0.4747
CV: 0.4712 ± 0.0115
OOF Macro F1: 0.4753

실험: A_kidney
Fold 1: Macro F1 = 0.4761
Fold 2: Macro F1 = 0.4518
Fold 3: Macro F1 = 0.4559
Fold 4: Macro F1 = 0.4832
Fold 5: Macro F1 = 0.4722
CV: 0.4678 ± 0.0120
OOF Macro F1: 0.4718

실험: B_brain
Fold 1: Macro F1 = 0.4763
Fold 2: Macro F1 = 0.4521
Fold 3: Macro F1 = 0.4549
Fold 4: Macro F1 = 0.4861
Fold 5: Macro F1 = 0.4630
CV: 0.4665 ± 0.0129
OOF Macro F1: 0.4705

실험: C_all
Fold 1: Macro F1 = 0.4737
Fold 2: Macro F1 = 0.4520
Fold 3: Macro F1 = 0.4666
Fold 4: Macro F1 = 0.4903
Fold 5: Macro F1 = 0.4690
CV: 0.4703 ± 0.0123
OOF Macro F1: 0.4750

=== 3차 실험 최종 결과 ===
experiment  cv_mean_macro_f1  cv_std  oof_macro_f1  delta_vs_baseline
  Baseline            0.4712  0.0115        0.4753             0.0000
  A_kidney            0.4678  0.0120        0.4718            -0.0034
   B_brain            0.4665  0.0

In [30]:
# 3차 실험 STEP 8. 관심 암종별 F1 / Recall 비교

from sklearn.metrics import classification_report
import pandas as pd

target_classes = ["KIPAN", "KIRC", "GBMLGG", "LGG"]

rows = []

for name, pred in oof_preds_pos.items():

    report = classification_report(
        y_dev,
        pred,
        labels=np.arange(len(le.classes_)),
        target_names=le.classes_,
        output_dict=True,
        zero_division=0
    )

    for cancer in target_classes:
        rows.append({
            "experiment": name,
            "cancer": cancer,
            "f1": report[cancer]["f1-score"],
            "recall": report[cancer]["recall"]
        })

class_results_pos = pd.DataFrame(rows)

print("=== 3차 실험 클래스별 F1 ===")
print(
    class_results_pos.pivot(
        index="cancer",
        columns="experiment",
        values="f1"
    ).round(4).to_string()
)

print("\n=== 3차 실험 클래스별 Recall ===")
print(
    class_results_pos.pivot(
        index="cancer",
        columns="experiment",
        values="recall"
    ).round(4).to_string()
)

=== 3차 실험 클래스별 F1 ===
experiment  A_kidney  B_brain  Baseline   C_all
cancer                                         
GBMLGG        0.5082   0.5164    0.5144  0.5120
KIPAN         0.4024   0.4094    0.4106  0.4102
KIRC          0.2931   0.2699    0.2945  0.2830
LGG           0.4390   0.4334    0.4492  0.4383

=== 3차 실험 클래스별 Recall ===
experiment  A_kidney  B_brain  Baseline   C_all
cancer                                         
GBMLGG        0.5447   0.5556    0.5556  0.5501
KIPAN         0.4976   0.5049    0.5097  0.5097
KIRC          0.2322   0.2097    0.2322  0.2210
LGG           0.3934   0.3825    0.3989  0.3880


In [31]:
# 3차 실험 STEP 9. 목표 암종 간 오분류 건수 비교

from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np

pairs = [
    ("KIPAN", "KIRC"),
    ("KIRC", "KIPAN"),
    ("GBMLGG", "LGG"),
    ("LGG", "GBMLGG")
]

rows = []

for name, pred in oof_preds_pos.items():

    cm = confusion_matrix(
        y_dev,
        pred,
        labels=np.arange(len(le.classes_))
    )

    class_to_idx = {
        label: i for i, label in enumerate(le.classes_)
    }

    row = {"experiment": name}

    for true_class, pred_class in pairs:
        row[f"{true_class} → {pred_class}"] = cm[
            class_to_idx[true_class],
            class_to_idx[pred_class]
        ]

    rows.append(row)

confusion_pos_df = pd.DataFrame(rows)

print("=== 3차 실험 주요 암종 간 오분류 ===")
print(confusion_pos_df.to_string(index=False))

=== 3차 실험 주요 암종 간 오분류 ===
experiment  KIPAN → KIRC  KIRC → KIPAN  GBMLGG → LGG  LGG → GBMLGG
  Baseline            73           132            67            95
  A_kidney            75           133            68            94
   B_brain            71           137            67            98
     C_all            73           136            67            96


# 5. 기록 (subtype_feature, subtype_feature_full)

In [19]:
# STEP 17. 기존 실험 기록 CSV 확인

import pandas as pd

csv_path = ROOT / "experiments" / "experiment_results.csv"

print("CSV 경로:", csv_path)
print("파일 존재:", csv_path.exists())

if csv_path.exists():
    existing_df = pd.read_csv(csv_path)

    print("\n기존 컬럼:")
    print(existing_df.columns.tolist())

    print("\n기존 실험 수:", len(existing_df))

    print("\n최근 5개 기록:")
    display(existing_df.tail(5))

else:
    print("\n기존 CSV가 없습니다. 새 파일을 생성해야 합니다.")

CSV 경로: /Users/admin/Documents/ah07-cancer-subtype/experiments/experiment_results.csv
파일 존재: False

기존 CSV가 없습니다. 새 파일을 생성해야 합니다.


In [20]:
# STEP 18. 1차 + 2차 실험 결과 CSV 생성

import pandas as pd
from pathlib import Path

csv_path = ROOT / "experiments" / "experiment_results.csv"

# 1차 실험: Feature E 단독 (전체 6,201건 CV)
first_results = [
    ("Baseline_E", 0.4833, 0.0108, 0.4842, 0.0000),
    ("A_kidney",   0.4831, 0.0137, 0.4846, -0.0001),
    ("B_brain",    0.4843, 0.0151, 0.4857, 0.0011),
    ("C_all",      0.4870, 0.0140, 0.4885, 0.0038),
]

# 2차 실험: E + Hotspot ×3 + Position-bin (개발 데이터 4,960건 CV)
second_results = [
    ("Baseline",  0.4724, 0.0189, 0.4774, 0.0000),
    ("D_kidney",  0.4649, 0.0083, 0.4692, -0.0074),
    ("E_brain",   0.4668, 0.0141, 0.4709, -0.0056),
    ("F_all",     0.4692, 0.0116, 0.4734, -0.0031),
]

records = []

for name, mean, std, oof, delta in first_results:
    records.append({
        "experiment_id": f"subtype_ablation_01_{name}",
        "notebook": "11_subtype_feature_ablation.ipynb",
        "cv_scope": "full_train_6201",
        "model": "XGBoost_depth3",
        "feature_base": "E",
        "experiment": name,
        "cv_mean_macro_f1": mean,
        "cv_std": std,
        "oof_macro_f1": oof,
        "delta_vs_baseline": delta,
        "holdout_used": True,
        "notes": "전체 train CV; 고정 holdout 행이 CV에 포함됨"
    })

for name, mean, std, oof, delta in second_results:
    records.append({
        "experiment_id": f"subtype_ablation_02_{name}",
        "notebook": "12_subtype_feature_full_ablation.ipynb",
        "cv_scope": "dev_only_4960",
        "model": "XGBoost_depth3",
        "feature_base": "E+Hotspot3+PositionBin50",
        "experiment": name,
        "cv_mean_macro_f1": mean,
        "cv_std": std,
        "oof_macro_f1": oof,
        "delta_vs_baseline": delta,
        "holdout_used": False,
        "notes": "고정 holdout 제외; Position-bin은 fold별 fit"
    })

new_df = pd.DataFrame(records)

# 기존 파일이 생겼다면 내용 보존 후 추가
if csv_path.exists():
    existing_df = pd.read_csv(csv_path)

    combined_df = pd.concat(
        [existing_df, new_df],
        ignore_index=True
    )

    combined_df = combined_df.drop_duplicates(
        subset=["experiment_id"],
        keep="last"
    )
else:
    combined_df = new_df

csv_path.parent.mkdir(parents=True, exist_ok=True)
combined_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("CSV 저장 완료:", csv_path)
print("전체 기록 수:", len(combined_df))

display(combined_df)

CSV 저장 완료: /Users/admin/Documents/ah07-cancer-subtype/experiments/experiment_results.csv
전체 기록 수: 8


,experiment_id,notebook,cv_scope,model,feature_base,experiment,cv_mean_macro_f1,cv_std,oof_macro_f1,delta_vs_baseline,holdout_used,notes
0,subtype_ablation_01_Baseline_E,11_subtype_feature_ablation.ipynb,full_train_6201,XGBoost_depth3,E,Baseline_E,0.4833,0.0108,0.4842,0.0000,True,전체 train CV; 고정 holdout 행이 CV에 포함됨
1,subtype_ablation_01_A_kidney,11_subtype_feature_ablation.ipynb,full_train_6201,XGBoost_depth3,E,A_kidney,0.4831,0.0137,0.4846,-0.0001,True,전체 train CV; 고정 holdout 행이 CV에 포함됨
2,subtype_ablation_01_B_brain,11_subtype_feature_ablation.ipynb,full_train_6201,XGBoost_depth3,E,B_brain,0.4843,0.0151,0.4857,0.0011,True,전체 train CV; 고정 holdout 행이 CV에 포함됨
3,subtype_ablation_01_C_all,11_subtype_feature_ablation.ipynb,full_train_6201,XGBoost_depth3,E,C_all,0.4870,0.0140,0.4885,0.0038,True,전체 train CV; 고정 holdout 행이 CV에 포함됨
4,subtype_ablation_02_Baseline,12_subtype_feature_full_ablation.ipynb,dev_only_4960,XGBoost_depth3,E+Hotspot3+PositionBin50,Baseline,0.4724,0.0189,0.4774,0.0000,False,고정 holdout 제외; Position-bin은 fold별 fit
5,subtype_ablation_02_D_kidney,12_subtype_feature_full_ablation.ipynb,dev_only_4960,XGBoost_depth3,E+Hotspot3+PositionBin50,D_kidney,0.4649,0.0083,0.4692,-0.0074,False,고정 holdout 제외; Position-bin은 fold별 fit
6,subtype_ablation_02_E_brain,12_subtype_feature_full_ablation.ipynb,dev_only_4960,XGBoost_depth3,E+Hotspot3+PositionBin50,E_brain,0.4668,0.0141,0.4709,-0.0056,False,고정 holdout 제외; Position-bin은 fold별 fit
7,subtype_ablation_02_F_all,12_subtype_feature_full_ablation.ipynb,dev_only_4960,XGBoost_depth3,E+Hotspot3+PositionBin50,F_all,0.4692,0.0116,0.4734,-0.0031,False,고정 holdout 제외; Position-bin은 fold별 fit


In [21]:
# STEP 20. CSV에 기록된 노트북 경로 수정

import pandas as pd

csv_path = ROOT / "experiments" / "experiment_results.csv"

df = pd.read_csv(csv_path)

df["notebook"] = df["notebook"].replace({
    "11_subtype_feature_ablation.ipynb":
        "notebooks/H_data/06_subtype_feature_ablation.ipynb",

    "12_subtype_feature_full_ablation.ipynb":
        "notebooks/H_data/07_subtype_feature_full_ablation.ipynb"
})

# 컬럼명 명확하게 변경
df = df.rename(columns={
    "holdout_used": "holdout_in_cv"
})

df.to_csv(csv_path, index=False, encoding="utf-8-sig")

display(df[[
    "experiment_id",
    "notebook",
    "holdout_in_cv"
]])

,experiment_id,notebook,holdout_in_cv
0,subtype_ablation_01_Baseline_E,notebooks/H_data/06_subtype_feature_ablation.i...,True
1,subtype_ablation_01_A_kidney,notebooks/H_data/06_subtype_feature_ablation.i...,True
2,subtype_ablation_01_B_brain,notebooks/H_data/06_subtype_feature_ablation.i...,True
3,subtype_ablation_01_C_all,notebooks/H_data/06_subtype_feature_ablation.i...,True
4,subtype_ablation_02_Baseline,notebooks/H_data/07_subtype_feature_full_ablat...,False
5,subtype_ablation_02_D_kidney,notebooks/H_data/07_subtype_feature_full_ablat...,False
6,subtype_ablation_02_E_brain,notebooks/H_data/07_subtype_feature_full_ablat...,False
7,subtype_ablation_02_F_all,notebooks/H_data/07_subtype_feature_full_ablat...,False


In [32]:
# 3차 실험 STEP 10. 실험 결과 CSV 업데이트

from pathlib import Path
import pandas as pd

csv_path = Path("../../experiments/experiment_results.csv")

assert csv_path.exists(), f"CSV 파일을 찾을 수 없음: {csv_path.resolve()}"

df_existing = pd.read_csv(csv_path)

new_rows = []

for _, row in results_pos_df.iterrows():

    experiment = row["experiment"]

    new_rows.append({
        "experiment_id": f"position_only_{experiment}",
        "notebook": "notebooks/H_data/07_subtype_feature_full_ablation.ipynb",
        "cv_scope": "dev_only_4960",
        "model": "XGBoost",
        "feature_base": "Feature E + Position-bin",
        "experiment": experiment,
        "cv_mean_macro_f1": row["cv_mean_macro_f1"],
        "cv_std": row["cv_std"],
        "oof_macro_f1": row["oof_macro_f1"],
        "delta_vs_baseline": row["delta_vs_baseline"],
        "holdout_in_cv": False,
        "notes": "No hotspot; 5-fold dev CV; position bins fitted within each training fold"
    })

df_new = pd.DataFrame(new_rows)

# 기존에 동일 실험이 있으면 새 결과로 교체
df_updated = pd.concat([
    df_existing[
        ~df_existing["experiment_id"].isin(df_new["experiment_id"])
    ],
    df_new
], ignore_index=True)

df_updated.to_csv(csv_path, index=False)

print("저장 경로:", csv_path.resolve())
print("전체 실험 수:", len(df_updated))
print("\n=== 추가된 3차 실험 결과 ===")
print(df_updated.tail(4).to_string(index=False))

저장 경로: /Users/admin/Documents/ah07-cancer-subtype/experiments/experiment_results.csv
전체 실험 수: 12

=== 추가된 3차 실험 결과 ===
         experiment_id                                                notebook      cv_scope   model             feature_base experiment  cv_mean_macro_f1   cv_std  oof_macro_f1  delta_vs_baseline  holdout_in_cv                                                                     notes
position_only_Baseline notebooks/H_data/07_subtype_feature_full_ablation.ipynb dev_only_4960 XGBoost Feature E + Position-bin   Baseline          0.471216 0.011488      0.475302           0.000000          False No hotspot; 5-fold dev CV; position bins fitted within each training fold
position_only_A_kidney notebooks/H_data/07_subtype_feature_full_ablation.ipynb dev_only_4960 XGBoost Feature E + Position-bin   A_kidney          0.467842 0.012032      0.471815          -0.003374          False No hotspot; 5-fold dev CV; position bins fitted within each training fold
 position_only_B_brain 